# Riepilogo Comparativo — Approcci Giornaliero vs Settimanale

🇮🇹 [🇬🇧](08_comparison_summary.ipynb)

## Predizione del Rischio di Infortunio nei Runner — Replica di Lovdal et al. (2021)

Questo notebook consolida i risultati dei notebook 04–07 in un **confronto unificato
giornaliero vs settimanale**. Non riaddestra alcun modello — carica i modelli XGBoost
salvati e ricalcola metriche, valori SHAP e confronti con benchmark dinamicamente.

### Contenuti

1. Setup
2. Caricamento dati e predizioni
3. Tabella metriche affiancata
4. Confronto con i benchmark dello studio (Lovdal et al., 2021)
5. Sovrapposizione curve ROC e PR
6. Confronto importanza feature (SHAP)
7. Raccomandazione finale
8. Limitazioni e lavori futuri
9. Riepilogo

### Benchmark dello studio

| Approccio | AUC-ROC (studio) | Modello |
|-----------|------------------|---------|
| Giornaliero | **0.724** | XGBoost con bagging (100 bag) |
| Settimanale | **0.678** | XGBoost con bagging (100 bag) |

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Ensure src/ is importable regardless of the notebook working directory
def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() or (candidate / "pyproject.toml").exists():
            return candidate
    return start

PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import INJURY_COL, RANDOM_SEED
from src.data_loading import get_feature_columns
from src.interpretability.shap_analysis import (
    compute_shap_values,
    get_shap_importance_dict,
    get_top_features,
)
from src.modeling.evaluate import (
    compute_metrics,
    create_comparison_table,
    find_optimal_threshold,
    plot_pr_curves,
    plot_roc_curves,
)
from src.preprocessing.io import load_model, load_splits
from src.utils.logging_config import setup_logging
from src.utils.plotting import PALETTE, save_figure, set_style
from src.utils.reproducibility import set_global_seed

# --- Reproducibility & style ---
set_global_seed(RANDOM_SEED)
setup_logging()
set_style()

# --- Paper benchmarks ---
PAPER_BENCHMARK_DAY: float = 0.724
PAPER_BENCHMARK_WEEK: float = 0.678

---

## 2. Caricamento Dati e Predizioni

Carichiamo i modelli XGBoost ottimizzati salvati nei notebook 04 e 05, insieme
agli split preprocessati di train e test. Per ogni approccio generiamo le probabilità
predette e troviamo la soglia di classificazione ottimale sulle predizioni del training
set (massimizzando F1), poi la applichiamo al test set per evitare data leakage.

In [ ]:
# --- Day approach ---
day_model = load_model("day_best_model")
day_train, day_test = load_splits(prefix="day")
feature_cols_day = get_feature_columns(day_test)
X_train_day = day_train[feature_cols_day]
y_train_day = day_train[INJURY_COL]
X_test_day = day_test[feature_cols_day]
y_test_day = day_test[INJURY_COL]

# Threshold selected on train predictions to avoid test leakage
y_prob_train_day = day_model.predict_proba(X_train_day)[:, 1]
threshold_day = find_optimal_threshold(np.array(y_train_day), y_prob_train_day, metric="f1")
y_prob_day = day_model.predict_proba(X_test_day)[:, 1]
y_pred_day = (y_prob_day >= threshold_day).astype(int)

# --- Week approach ---
week_model = load_model("week_best_model")
week_train, week_test = load_splits(prefix="week")
feature_cols_week = get_feature_columns(week_test)
X_train_week = week_train[feature_cols_week]
y_train_week = week_train[INJURY_COL]
X_test_week = week_test[feature_cols_week]
y_test_week = week_test[INJURY_COL]

# Threshold selected on train predictions to avoid test leakage
y_prob_train_week = week_model.predict_proba(X_train_week)[:, 1]
threshold_week = find_optimal_threshold(np.array(y_train_week), y_prob_train_week, metric="f1")
y_prob_week = week_model.predict_proba(X_test_week)[:, 1]
y_pred_week = (y_prob_week >= threshold_week).astype(int)

# --- Summary ---
print("Day approach:")
print(f"  Test set: {X_test_day.shape[0]:,} rows x {X_test_day.shape[1]} features")
print(f"  Injury rate: {y_test_day.mean():.2%}")
print(f"  Train-selected threshold: {threshold_day:.2f}")
print(f"  Predicted positives: {y_pred_day.sum()} ({y_pred_day.mean():.2%})")

print("\nWeek approach:")
print(f"  Test set: {X_test_week.shape[0]:,} rows x {X_test_week.shape[1]} features")
print(f"  Injury rate: {y_test_week.mean():.2%}")
print(f"  Train-selected threshold: {threshold_week:.2f}")
print(f"  Predicted positives: {y_pred_week.sum()} ({y_pred_week.mean():.2%})")

---

## 3. Tabella Metriche Affiancata

Confronto completo dei modelli XGBoost ottimizzati sul test set riservato.
Entrambi i modelli sono valutati alle rispettive soglie ottimali selezionate
sul training (massimizzando F1 sulle predizioni di training, applicate al test).
Una colonna delta evidenzia quale approccio performa meglio su ogni metrica.

In [ ]:
# Compute metrics for both approaches at optimal thresholds
day_metrics = compute_metrics(np.array(y_test_day), y_pred_day, y_prob_day)
week_metrics = compute_metrics(np.array(y_test_week), y_pred_week, y_prob_week)
day_xgb_label = "Day XGBoost"
week_xgb_label = "Week XGBoost"

# Build comparison table
comparison = create_comparison_table({
    day_xgb_label: day_metrics,
    week_xgb_label: week_metrics,
})

# Select key metrics and add delta (computed from unrounded dicts for accuracy)
key_metrics = ["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]
cmp_table = comparison[key_metrics].copy()
day_metric_values = pd.Series({metric: day_metrics[metric] for metric in key_metrics})
week_metric_values = pd.Series({metric: week_metrics[metric] for metric in key_metrics})
cmp_table.loc["Delta (Day − Week)"] = (day_metric_values - week_metric_values).round(4)

print(f"Test Set Metrics — Day vs Week (thresholds: day={threshold_day:.2f}, week={threshold_week:.2f}):\n")
print(cmp_table.to_string())

# Highlight winners
print("\n\nPer-metric winner:")
for metric in key_metrics:
    day_val = day_metrics[metric]
    week_val = week_metrics[metric]
    # Lower is better for Brier score
    if metric == "brier_score":
        winner = "Day" if day_val <= week_val else "Week"
    else:
        winner = "Day" if day_val >= week_val else "Week"
    print(f"  {metric:>15s}: {winner} ({day_val:.4f} vs {week_val:.4f})")

In [ ]:
# Grouped bar chart: day vs week on key metrics
display_metrics = ["auc_roc", "auc_pr", "recall", "precision", "f1", "brier_score"]
labels = ["AUC-ROC", "AUC-PR", "Recall", "Precision", "F1", "Brier Score"]

day_vals = [day_metrics[m] for m in display_metrics]
week_vals = [week_metrics[m] for m in display_metrics]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars_day = ax.bar(x - width / 2, day_vals, width, label="Day", color=PALETTE["primary"], alpha=0.85)
bars_week = ax.bar(x + width / 2, week_vals, width, label="Week", color=PALETTE["secondary"], alpha=0.85)

# Value labels
for bars in [bars_day, bars_week]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=9, fontweight="bold")

ax.set_ylabel("Score")
ax.set_title("Day vs Week — Test Set Metrics Comparison", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
sns.despine()

save_figure(fig, "08_metrics_comparison_day_vs_week", subdir="comparison", close=False)
plt.show()
plt.close(fig)

---

## 4. Confronto con i Benchmark dello Studio

Lovdal et al. (2021) hanno riportato AUC-ROC di **0.724** (giornaliero) e **0.678**
(settimanale) usando un **ensemble XGBoost con bagging** (100 bag). I nostri risultati
usano un singolo modello XGBoost ottimizzato — un'architettura più semplice che ci si
aspetta produca un AUC-ROC leggermente inferiore.

In [ ]:
# Benchmark comparison table
our_day_auc = day_metrics["auc_roc"]
our_week_auc = week_metrics["auc_roc"]
our_xgb_label = "Ours (single XGBoost)"
paper_xgb_label = "Paper (bagged XGBoost)"

benchmark_data = {
    "Approach": ["Day", "Day", "Week", "Week"],
    "Source": [our_xgb_label, paper_xgb_label, our_xgb_label, paper_xgb_label],
    "AUC-ROC": [our_day_auc, PAPER_BENCHMARK_DAY, our_week_auc, PAPER_BENCHMARK_WEEK],
}

benchmark_df = pd.DataFrame(benchmark_data)
benchmark_df["Delta vs Paper"] = benchmark_df.apply(
    lambda row: row["AUC-ROC"] - (PAPER_BENCHMARK_DAY if row["Approach"] == "Day" else PAPER_BENCHMARK_WEEK),
    axis=1,
)
benchmark_df["% of Paper"] = benchmark_df.apply(
    lambda row: row["AUC-ROC"] / (PAPER_BENCHMARK_DAY if row["Approach"] == "Day" else PAPER_BENCHMARK_WEEK) * 100,
    axis=1,
)

print("Paper Benchmark Comparison:\n")
print(benchmark_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Bar chart: our AUC-ROC vs paper benchmarks
approaches = ["Day", "Week"]
our_aucs = [our_day_auc, our_week_auc]
paper_aucs = [PAPER_BENCHMARK_DAY, PAPER_BENCHMARK_WEEK]

x = np.arange(len(approaches))
width = 0.3

fig, ax = plt.subplots(figsize=(8, 5))
bars_ours = ax.bar(x - width / 2, our_aucs, width, label="Ours (single XGBoost)",
                   color=PALETTE["primary"], alpha=0.85)
bars_paper = ax.bar(x + width / 2, paper_aucs, width, label="Paper (bagged XGBoost)",
                    color=PALETTE["neutral"], alpha=0.7, hatch="//", edgecolor="white")

# Value labels
for bars in [bars_ours, bars_paper]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{bar.get_height():.3f}", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("AUC-ROC")
ax.set_title("AUC-ROC — Our Results vs Paper Benchmarks", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(approaches)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
sns.despine()

save_figure(fig, "08_paper_benchmark_comparison", subdir="comparison", close=False)
plt.show()
plt.close(fig)

### Perché i nostri risultati differiscono dallo studio

Diverse differenze metodologiche spiegano il gap di AUC-ROC:

1. **Nessun ensemble con bagging**: lo studio ha usato un ensemble XGBoost a 100 bag
   che media le predizioni su 100 modelli addestrati indipendentemente, riducendo la
   varianza. Noi abbiamo usato un singolo XGBoost ottimizzato — una scelta deliberata
   di semplicità.

2. **Separazione più rigorosa degli atleti (ADR-006)**: applichiamo `GroupKFold` e
   `GroupShuffleSplit` per Athlete ID ovunque. La strategia esatta di split dello studio
   potrebbe differire, e una separazione più rigorosa riduce tipicamente la performance
   apparente (nessun information leakage tra atleti).

3. **Budget per l'ottimizzazione degli iperparametri**: abbiamo usato `RandomizedSearchCV`
   con 30 iterazioni per la fattibilità locale. Lo studio probabilmente ha usato una
   ricerca più esaustiva o default informati dal dominio.

4. **Gestione sentinella (ADR-007)**: abbiamo sostituito i valori sentinella (−0.01)
   con 0.0 per rappresentare i giorni di riposo. Trattamenti diversi di questi valori
   influenzano le soglie delle feature apprese.

5. **Binarizzazione target settimanale (ADR-002)**: abbiamo binarizzato alla soglia 0.5,
   coerente con lo studio, ma i dettagli implementativi nella conversione da continuo a
   binario possono introdurre piccole differenze.

Nonostante queste differenze, i nostri risultati sono in un range ragionevole rispetto
ai benchmark dello studio, validando la metodologia core della replica.

---

## 5. Sovrapposizione Curve ROC e PR

Sovrapporre le curve ROC/PR giornaliere e settimanali sugli stessi assi permette un
confronto visivo diretto della performance discriminativa su tutte le soglie di
classificazione.

In [ ]:
# Combined ROC curves — day and week on same axes
combined_predictions = {
    day_xgb_label: (np.array(y_test_day), y_prob_day),
    week_xgb_label: (np.array(y_test_week), y_prob_week),
}

fig_roc = plot_roc_curves(combined_predictions, save_path=None)
fig_roc.axes[0].annotate(
    f"Paper benchmarks: Day={PAPER_BENCHMARK_DAY}, Week={PAPER_BENCHMARK_WEEK}",
    xy=(0.45, 0.05), xycoords="axes fraction", fontsize=10, fontstyle="italic", color=PALETTE["negative"],
)
save_figure(fig_roc, "08_roc_curves_combined", subdir="comparison", close=False)
plt.show()
plt.close(fig_roc)

In [ ]:
# Combined PR curves — day and week on same axes
# Note: PR curves are prevalence-dependent; the two test sets may have
# slightly different positive rates, so compare shapes with caution.
day_pos_rate = float(np.mean(combined_predictions["Day XGBoost"][0]))
week_pos_rate = float(np.mean(combined_predictions["Week XGBoost"][0]))

fig_pr = plot_pr_curves(combined_predictions, save_path=None)
fig_pr.axes[0].annotate(
    f"Positive rate — Day: {day_pos_rate:.3f}, Week: {week_pos_rate:.3f}\n"
    "PR curves are prevalence-dependent; compare with caution.",
    xy=(0.02, 0.02), xycoords="axes fraction", fontsize=9, fontstyle="italic",
    color=PALETTE["negative"],
    bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)
save_figure(fig_pr, "08_pr_curves_combined", subdir="comparison", close=False)
plt.show()
plt.close(fig_pr)

---

## 6. Confronto Importanza Feature (SHAP)

L'importanza delle feature basata su SHAP rivela quali feature di allenamento guidano
le predizioni in ogni approccio temporale. Poiché gli approcci giornaliero e settimanale
usano insiemi di feature diversi (7 giorni x 10 feature vs 3 settimane x 22 feature + 3
rapporti), confrontiamo a due livelli:

1. **Livello feature grezza**: top feature per approccio (nomi di feature diversi)
2. **Livello concettuale**: rimuovendo i prefissi temporali per identificare quali
   metriche di allenamento base (volume, intensità, recupero, ecc.) sono consistentemente
   importanti indipendentemente dalla granularità temporale

In [ ]:
# Compute SHAP values for both approaches
shap_values_day = compute_shap_values(day_model, X_test_day)
shap_values_week = compute_shap_values(week_model, X_test_week)

# Top 10 features per approach
top10_day = get_top_features(shap_values_day, n=10)
top10_week = get_top_features(shap_values_week, n=10)

print("Top 10 features by mean |SHAP value|:\n")
print(f"  {'Day Approach':<35s}  {'Week Approach':<35s}")
print(f"  {'─' * 35}  {'─' * 35}")
for i in range(10):
    d = top10_day[i] if i < len(top10_day) else ""
    w = top10_week[i] if i < len(top10_week) else ""
    print(f"  {i + 1:>2d}. {d:<32s}  {i + 1:>2d}. {w:<32s}")

In [ ]:
# Two-panel chart: top 15 SHAP features per approach
shap_imp_day = get_shap_importance_dict(shap_values_day)
shap_imp_week = get_shap_importance_dict(shap_values_week)

top_n = 15

# Sort by importance, take top N
day_sorted = sorted(shap_imp_day.items(), key=lambda x: x[1], reverse=True)[:top_n]
week_sorted = sorted(shap_imp_week.items(), key=lambda x: x[1], reverse=True)[:top_n]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Day panel
day_names = [x[0] for x in reversed(day_sorted)]
day_values = [x[1] for x in reversed(day_sorted)]
ax1.barh(day_names, day_values, color=PALETTE["primary"], alpha=0.85)
ax1.set_xlabel("Mean |SHAP value|")
ax1.set_title("Day Approach — Top 15 Features", fontweight="bold")

# Week panel
week_names = [x[0] for x in reversed(week_sorted)]
week_values = [x[1] for x in reversed(week_sorted)]
ax2.barh(week_names, week_values, color=PALETTE["secondary"], alpha=0.85)
ax2.set_xlabel("Mean |SHAP value|")
ax2.set_title("Week Approach — Top 15 Features", fontweight="bold")

fig.suptitle("SHAP Feature Importance — Day vs Week", fontweight="bold", fontsize=14, y=1.02)
fig.tight_layout()

save_figure(fig, "08_shap_importance_day_vs_week", subdir="comparison", close=False)
plt.show()
plt.close(fig)

In [ ]:
# Concept-level overlap: strip temporal prefix and aggregate by base feature type
def extract_base_feature(feature_name: str) -> str:
    """Strip temporal prefix (day_N_ or week_N_) to get the base feature type."""
    return re.sub(r"^(day|week)_\d+_", "", feature_name)


def aggregate_importance_by_type(importance: dict[str, float]) -> dict[str, float]:
    """Sum SHAP importance across temporal windows for each base feature type."""
    agg: dict[str, float] = {}
    for feat, val in importance.items():
        base = extract_base_feature(feat)
        agg[base] = agg.get(base, 0.0) + val
    return agg


day_by_type = aggregate_importance_by_type(shap_imp_day)
week_by_type = aggregate_importance_by_type(shap_imp_week)

# Common base feature types
common_types = sorted(set(day_by_type) & set(week_by_type))

print(f"Day unique feature types: {len(day_by_type)}")
print(f"Week unique feature types: {len(week_by_type)}")
print(f"Common feature types: {len(common_types)}")

if common_types:
    # Normalize to percentages for fair comparison
    day_total = sum(day_by_type[t] for t in common_types)
    week_total = sum(week_by_type[t] for t in common_types)

    day_pct = {t: day_by_type[t] / day_total * 100 for t in common_types}
    week_pct = {t: week_by_type[t] / week_total * 100 for t in common_types}

    # Sort by average importance
    sorted_types = sorted(common_types, key=lambda t: (day_pct[t] + week_pct[t]) / 2, reverse=True)

    print("\nCommon feature types ranked by average importance:\n")
    print(f"  {'Feature type':<35s}  {'Day %':>8s}  {'Week %':>8s}")
    print(f"  {'─' * 35}  {'─' * 8}  {'─' * 8}")
    for t in sorted_types:
        print(f"  {t:<35s}  {day_pct[t]:>7.1f}%  {week_pct[t]:>7.1f}%")

---

## 7. Raccomandazione Finale

In [ ]:
# Programmatic summary: which approach wins on each dimension
print("=" * 60)
print("RECOMMENDATION SUMMARY")
print("=" * 60)

# Performance comparison
metrics_to_compare = {
    "AUC-ROC": ("auc_roc", "higher"),
    "AUC-PR": ("auc_pr", "higher"),
    "Recall": ("recall", "higher"),
    "Precision": ("precision", "higher"),
    "F1": ("f1", "higher"),
    "Brier Score": ("brier_score", "lower"),
}

day_wins = 0
week_wins = 0

print("\n1. PERFORMANCE (test set, train-selected thresholds)")
print(f"   {'Metric':<15s}  {'Day':>8s}  {'Week':>8s}  {'Winner':>8s}")
print(f"   {'─' * 15}  {'─' * 8}  {'─' * 8}  {'─' * 8}")

for label, (key, direction) in metrics_to_compare.items():
    d = day_metrics[key]
    w = week_metrics[key]
    if direction == "higher":
        winner = "Day" if d >= w else "Week"
    else:
        winner = "Day" if d <= w else "Week"
    if winner == "Day":
        day_wins += 1
    else:
        week_wins += 1
    print(f"   {label:<15s}  {d:>8.4f}  {w:>8.4f}  {winner:>8s}")

print(f"\n   Score: Day {day_wins} — Week {week_wins}")

# Paper proximity
day_pct_paper = our_day_auc / PAPER_BENCHMARK_DAY * 100
week_pct_paper = our_week_auc / PAPER_BENCHMARK_WEEK * 100
print("\n2. PAPER BENCHMARK PROXIMITY")
print(f"   Day:  {our_day_auc:.4f} / {PAPER_BENCHMARK_DAY} = {day_pct_paper:.1f}% of paper")
print(f"   Week: {our_week_auc:.4f} / {PAPER_BENCHMARK_WEEK} = {week_pct_paper:.1f}% of paper")

print("\n3. INTERPRETABILITY (from notebook 06)")
print(f"   Day:  {len(feature_cols_day)} features (7 days x 10 metrics) — finer temporal resolution")
print(f"   Week: {len(feature_cols_week)} features (3 weeks x 22+ metrics) — includes relative km ratios")

print("\n4. FAIRNESS (from notebook 07)")
print("   Both approaches audited across 3 proxy grouping strategies.")
print("   Cross-comparison in reports/figures/fairness/07_day_vs_week_disparity_comparison.png")

print("\n" + "=" * 60)

### Raccomandazione

Basandoci sul confronto multi-dimensionale sopra:

**1. Performance discriminativa**: l'approccio giornaliero raggiunge consistentemente
un AUC-ROC e AUC-PR più alti, coerente con i risultati dello studio. La granularità
giornaliera cattura i picchi acuti di carico che l'aggregazione settimanale attenua.

**2. Utilità clinica**: nella prevenzione degli infortuni, il **recall** (rilevare gli
infortuni reali) è più importante della precision (evitare falsi allarmi). Un infortunio
mancato (falso negativo) ha un costo più alto di un segnale di cautela non necessario
(falso positivo). L'approccio con recall più alto a precision ragionevole dovrebbe essere
preferito per il deployment.

**3. Interpretabilità**: entrambi gli approcci producono pattern SHAP interpretabili.
L'approccio giornaliero rivela gli effetti acuti del carico giornaliero; l'approccio
settimanale cattura l'accumulo del carico a lungo termine e i rapporti acuto-cronico —
entrambi sono significativi per scienziati dello sport e allenatori.

**4. Fairness**: l'analisi con gruppi proxy (notebook 07) ha mostrato che entrambi gli
approcci hanno profili di fairness simili. Nessun approccio è sistematicamente più equo
dell'altro attraverso i raggruppamenti per volume, storia infortuni e densità dati.

**5. Deployment pratico**:
- **Approccio giornaliero**: fornisce avvisi più tempestivi (predizioni giornaliere),
  permette aggiustamenti dell'allenamento in giornata, risoluzione temporale più alta
- **Approccio settimanale**: predizioni più stabili (meno rumore da fluttuazioni
  giornaliere), include rapporti di carico acuto-cronico (un concetto ben validato nella
  scienza dello sport), più facile per gli allenatori agire su piani settimanali

**6. Complessivo**: entrambi gli approcci sono validi. La scelta dipende dal contesto
operativo — i sistemi di monitoraggio giornaliero favoriscono l'approccio giornaliero,
mentre i flussi di lavoro di pianificazione settimanale favoriscono l'approccio
settimanale. Per un sistema generale di predizione del rischio di infortunio,
l'**approccio giornaliero** offre un leggero vantaggio in performance discriminativa
e risoluzione temporale.

---

## 8. Limitazioni e Lavori Futuri

### Limitazioni

1. **Nessun ensemble con bagging**: l'XGBoost a 100 bag dello studio raggiunge un
   AUC-ROC più alto attraverso la riduzione della varianza. La nostra architettura a
   modello singolo è più semplice ma meno performante.

2. **Coorte di atleti ridotta**: 74 atleti totali, con circa 15 nel test set. I risultati
   potrebbero non generalizzare a popolazioni più ampie o diverse (es. runner amatoriali,
   sport diversi).

3. **Nessuna validazione esterna**: valutiamo su uno split riservato dello stesso dataset
   usato da Lovdal et al. La vera generalizzabilità richiede test su un dataset
   indipendente.

4. **Grave sbilanciamento delle classi**: l'approccio giornaliero ha solo l'1.36% di
   campioni positivi. Questo rende la calibrazione inaffidabile e metriche come
   precision e F1 altamente sensibili alla scelta della soglia.

5. **Nessuna validazione temporale**: usiamo GroupKFold (ADR-006) invece di
   TimeSeriesSplit. Sebbene questo prevenga il leakage tra atleti, non testa la
   capacità del modello di predire infortuni futuri da dati passati in senso
   strettamente temporale.

6. **Nessuna fairness demografica**: senza età, sesso o altri attributi protetti,
   l'analisi con gruppi proxy fornisce garanzie di fairness limitate. Bias sistematici
   tra gruppi demografici non possono essere rilevati o esclusi.

7. **Dataset singolo, sport singolo**: i risultati sono specifici per runner d'élite
   olandesi (2012–2019). La trasferibilità ad altri sport, livelli o sistemi di
   monitoraggio è sconosciuta.

### Lavori futuri

1. **Ensemble XGBoost con bagging**: implementare architettura a 100 bag per colmare
   il gap con i benchmark dello studio e quantificare la riduzione della varianza
2. **Approcci deep learning**: modelli LSTM o Transformer per pattern di serie temporali,
   potenzialmente catturando dipendenze a lungo raggio
3. **Raccolta dati demografici**: abilitare un audit di fairness appropriato sugli
   attributi protetti
4. **Validazione esterna**: testare su popolazioni di runner indipendenti (paesi diversi,
   livelli di competizione, sistemi di monitoraggio)
5. **Design di studio prospettico**: addestrare su dati storici (es. 2012–2017), validare
   su stagioni future (2018–2019) per testare la generalizzazione temporale
6. **Miglioramento della calibrazione**: Platt scaling o regressione isotonica per
   produrre probabilità di infortunio affidabili (non solo ranking)
7. **Dashboard interattiva**: deployment su Looker Studio per il monitoraggio in tempo
   reale da parte di allenatori e scienziati dello sport (Fase 8 differita)

---

## 9. Riepilogo

### Flusso della pipeline

```
data/processed/day_best_model.pkl + week_best_model.pkl
  → load_model()
data/processed/day_test.parquet + week_test.parquet
  → load_splits()
  → compute_metrics() [confronto affiancato]
  → find_optimal_threshold() [ottimizzazione F1 per-approccio]
  → plot_roc_curves() + plot_pr_curves() [curve sovrapposte]
  → compute_shap_values() [confronto importanza feature]
  → Analisi di sovrapposizione a livello concettuale [tipi di feature base]
  → Tutte le figure salvate in reports/figures/comparison/
```

### Figure generate

Tutte le figure salvate in `reports/figures/comparison/`:
- 1 grafico a barre metriche affiancate (giornaliero vs settimanale)
- 1 grafico confronto benchmark dello studio
- 1 sovrapposizione curve ROC combinata
- 1 sovrapposizione curve PR combinata
- 1 grafico SHAP importanza a due pannelli

**Totale: 5 figure**

### Risultati principali

1. L'**approccio giornaliero supera quello settimanale** nelle metriche discriminative
   (AUC-ROC, AUC-PR), coerente con i risultati dello studio. La granularità giornaliera
   cattura i segnali acuti di carico che l'aggregazione settimanale attenua.

2. **Entrambi gli approcci sono sotto i benchmark dello studio**, principalmente per
   l'uso di un singolo modello XGBoost vs l'ensemble a 100 bag dello studio, e una
   separazione più rigorosa a livello di atleta (GroupKFold/GroupShuffleSplit).

3. **L'importanza delle feature è consistente a livello concettuale**: volume di
   allenamento e indicatori soggettivi (sforzo, recupero) sono importanti in entrambi
   gli approcci, validando il loro ruolo come indicatori di rischio di infortunio.

4. **Entrambi gli approcci mostrano profili di fairness simili** tra i gruppi proxy —
   nessuno è sistematicamente più equo.

5. **La scelta tra approcci dipende dal contesto operativo**: giornaliero per il
   monitoraggio in tempo reale, settimanale per i flussi di lavoro di pianificazione.
   L'approccio giornaliero ha un leggero vantaggio complessivo.

6. **La replica valida la metodologia core**: nonostante le semplificazioni
   architetturali, gli stessi pattern nei dati e le stesse classifiche dei modelli
   emergono.